In [15]:
# Зареждане на набора от данни
from torch_geometric.datasets import Entities

dataset = Entities(root="data/Entities", name="AIFB")
data = dataset[0]

In [21]:
# Изследване на графа
import torch

print("Брой възли:", data.num_nodes)
print("Брой ребра:", data.num_edges)

print("Брой типове отношения:",
      torch.unique(data.edge_type).numel())

print("Размер на обучаващото множество:",
      data.train_idx.size(0))

print("Размер на тестовото множество:",
      data.test_idx.size(0))

print(data)
print(data.keys())

Брой възли: 8285
Брой ребра: 58086
Брой типове отношения: 90
Размер на обучаващото множество: 140
Размер на тестовото множество: 36
Data(edge_index=[2, 58086], edge_type=[58086], train_idx=[140], train_y=[140], test_idx=[36], test_y=[36], num_nodes=8285)
['edge_index', 'edge_type', 'test_idx', 'num_nodes', 'train_idx', 'test_y', 'train_y']


In [23]:
# Изграждане на модел RGCN
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import RGCNConv


class RGCN(nn.Module):

    def __init__(self, num_nodes, num_relations,
                 hidden_channels, num_classes):

        super().__init__()

        # Embedding за възлите
        self.embedding = nn.Embedding(
            num_nodes,
            hidden_channels)

        # RGCN слоеве
        self.conv1 = RGCNConv(
            hidden_channels,
            hidden_channels,
            num_relations)

        self.conv2 = RGCNConv(
            hidden_channels,
            hidden_channels,
            num_relations)

        # Класификатор
        self.classifier = nn.Linear(
            hidden_channels,
            num_classes)

    def forward(self, edge_index, edge_type):

        x = self.embedding.weight

        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)

        x = self.conv2(x, edge_index, edge_type)
        x = F.relu(x)

        out = self.classifier(x)

        return out

In [24]:
# Създване на модела
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RGCN(
    num_nodes=data.num_nodes,
    num_relations=int(data.edge_type.max()) + 1,
    hidden_channels=64,
    num_classes=dataset.num_classes
).to(device)

print(model)

RGCN(
  (embedding): Embedding(8285, 64)
  (conv1): RGCNConv(64, 64, num_relations=90)
  (conv2): RGCNConv(64, 64, num_relations=90)
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)


In [25]:
# Дефиниране на функцията на загуба и оптимизатора
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4)

In [26]:
# Обучение
model.train()

for epoch in range(100):

    optimizer.zero_grad()

    out = model(
        data.edge_index,
        data.edge_type
    )

    loss = criterion(
        out[data.train_idx],
        data.train_y
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Епоха {epoch+1:3d} | Загуба: {loss.item():.4f}")

Епоха  10 | Загуба: 0.0495
Епоха  20 | Загуба: 0.0042
Епоха  30 | Загуба: 0.0014
Епоха  40 | Загуба: 0.0009
Епоха  50 | Загуба: 0.0011
Епоха  60 | Загуба: 0.0014
Епоха  70 | Загуба: 0.0016
Епоха  80 | Загуба: 0.0017
Епоха  90 | Загуба: 0.0018
Епоха 100 | Загуба: 0.0017


In [27]:
# Оценяване
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

model.eval()

with torch.no_grad():

    out = model(
        data.edge_index,
        data.edge_type
    )

    pred = out.argmax(dim=1)

    y_true = data.test_y.cpu()
    y_pred = pred[data.test_idx].cpu()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")

Accuracy: 0.8889
Macro F1-score: 0.8751


In [28]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Брой параметри: {num_params}")

Брой параметри: 1276100
